# **Tracing & Extracting Power BI DAX Queries with SemPy**

# Check out the following blogposts for details about this notebook
- [Tracing Power BI Events with SemPy](https://bits2bi.com/2026/03/28/real-time-power-bi-monitoring-tracing-power-bi-events-with-fabric-notebooks/)
- [Tracing DAX Query Execution with SemPy](https://bits2bi.com/2026/04/02/real-time-power-bi-monitoring-tracing-dax-query-execution-with-fabric-notebooks/)
- [Extracting User-generated DAX Queries with SemPy](https://bits2bi.com/2026/04/09/real-time-power-bi-monitoring-extracting-user-generated-dax-queries-with-fabric-notebooks/)

### Prepare Environment

In [ ]:
# Import the Semantic Link Python library
import sempy.fabric as fabric

# Import other libraries and modules
from datetime import datetime, timedelta
import time

import pandas as pd

### Specify Targets Details

In [ ]:
# Declare and assign variables

## ........ PROVIDE THE NAME OR ID OF YOUR TARGET WORKSPACE, TARGET SEMANTIC MODEL, AND THE DAX QUERY IN FOLLOWING VARIABLES ........ ##

target_workspace = "<TARGET_WORKSPACE_NAME_OR_ID>"
target_semantic_model = "<TARGET_SEMANTIC_MODEL_NAME_OR_ID>"

target_dax_query = """
<DAX_QUERY_TO_RUN>
"""

### Tracing DAX Query Execution with Default Query Trace Schema

In [ ]:
# Default query trace schema

default_query_trace_schema = fabric.Trace.get_default_query_trace_schema()
display(default_query_trace_schema)

In [ ]:
# Execute the DAX query while trace is active and monitor the events with the default query trace schema

# Establish a connection to the XMLA endpoint of the specific workspace and semantic model
# Using 'with' ensures the connection is closed automatically even if the code fails

with fabric.create_trace_connection(workspace = target_workspace, dataset = target_semantic_model) as trace_connection:
    # Define a trace session on the server side using the provided event_schema (e.g., Query End)
    with trace_connection.create_trace(default_query_trace_schema, "SemPyDAXQueryTrace") as trace:

        # Start the trace. The argument (5) is a timeout in seconds to wait for the trace to initialize
        trace.start(5)

        ## Execute the DAX query while the trace is "listening" in the background
        dax_result = fabric.evaluate_dax(
            workspace = target_workspace, 
            dataset = target_semantic_model,
            dax_string = target_dax_query
        )

        display(dax_result)

        ### CRITICAL: Wait for the server to flush the remaining events into the log buffer before we stop the trace
        time.sleep(5)

        # Stop the server-side trace and download the captured event data into a pandas DataFrame
        trace_logs = trace.stop()

display(trace_logs)
#trace_logs.info()

### Tracing DAX Query Execution with Custom Event Schema

In [ ]:
# Discover all event types and their associated columns
# https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric.traceconnection?view=semantic-link-python#sempy-fabric-traceconnection-discover-event-schema

with fabric.create_trace_connection(workspace = target_workspace, dataset = target_semantic_model) as trace_connection:
    discover_events = trace_connection.discover_event_schema()

print(len(discover_events))
display(discover_events)

In [ ]:
# Define events to trace and their corresponding data columns

event_query_begin_columns = [
    "EventClass",
    "EventSubclass",
    "RequestID",
    "ApplicationName",
    "NTCanonicalUserName",
    "SessionID",
    "ActivityID",
    "DatabaseName",
    "CurrentTime",
    "StartTime",
    "TextData"
]

event_query_end_columns = [
    "EventClass",
    "EventSubclass",
    "RequestID",
    "ApplicationName",
    "NTCanonicalUserName",
    "SessionID",
    "ActivityID",
    "DatabaseName",
    "CurrentTime",
    "StartTime",
    "EndTime",
    "Duration",
    "CpuTime",
    "Success",
    "Error",
    "TextData"
]

event_execution_metrics_columns = [
    "EventClass",
    "RequestID",
    "ApplicationName",
    "ActivityID",
    "DatabaseName",
    "TextData"
]

# Define event schema
event_schema = {
    "QueryBegin": event_query_begin_columns,
    "QueryEnd": event_query_end_columns,
    "ExecutionMetrics": event_execution_metrics_columns
}

#display(event_schema)

In [ ]:
# Execute the DAX query while trace is active and monitor the events with the custom event schema

# Establish a connection to the XMLA endpoint of the specific workspace and semantic model
# Using 'with' ensures the connection is closed automatically even if the code fails

with fabric.create_trace_connection(workspace = target_workspace, dataset = target_semantic_model) as trace_connection:
    # Define a trace session on the server side using the provided event_schema (e.g., Query End)
    with trace_connection.create_trace(event_schema, "SemPyDAXQueryTrace") as trace:

        # Start the trace. The argument (5) is a timeout in seconds to wait for the trace to initialize
        trace.start(5)

        ## Execute the DAX query while the trace is "listening" in the background
        dax_result = fabric.evaluate_dax(
            workspace = target_workspace, 
            dataset = target_semantic_model,
            dax_string = target_dax_query
        )

        display(dax_result)

        ### CRITICAL: Wait for the server to flush the remaining events into the log buffer before we stop the trace
        time.sleep(5)

        # Stop the server-side trace and download the captured event data into a pandas DataFrame
        trace_logs = trace.stop()

display(trace_logs)
#trace_logs.info()

# Tracing User-Generated DAX Queries

### Bounded Trace

A bounded trace includes a well-defined exit condition that terminates the trace and collects the event data. The exit could be triggered by a timer, an event, or the status of an operation.

#### Duration-Bound Trace
Run the trace for a predefined duration (e.g., 5 minutes). Once the time window expires, the trace stops automatically.

In [ ]:
## Duration-Based Trace

# Run the trace for a specified duration and collect the event at the end of the interval

# Set the total duration in seconds
total_duration = 60
check_interval = 20
elapsed_time = 0

# Establish a connection to the XMLA endpoint of the specific workspace and semantic model
# Using 'with' ensures the connection is closed automatically even if the code fails

with fabric.create_trace_connection(workspace = target_workspace, dataset = target_semantic_model) as trace_connection:
    # Define a trace session on the server side using the provided event_schema (e.g., Query End)
    with trace_connection.create_trace(event_schema, "SemPyDAXQueryTrace") as trace:

        # Start the trace. The argument (5) is a delay in seconds to wait for the trace to initialize
        trace.start(5)
        print("Trace started. Monitoring activity...")

        # Loop until the total duration is reached
        while elapsed_time < total_duration:
            time.sleep(check_interval)
            elapsed_time += check_interval
            print(f"Elapsed time: {elapsed_time} seconds...")

        ### CRITICAL: Final buffer to ensure the server flushes the last events
        print("Finalizing trace and flushing buffer...")
        time.sleep(5)

        # Stop the trace and download the data
        trace_logs = trace.stop()
        print("Trace complete")

display(trace_logs)

### Unbounded Trace

An unbounded trace doesn't specify an exit condition.

<mark> ⚠️ **Note:** Remember to properly clean up your trace connection and session to avoid leaving open connections. </mark>

In [ ]:
## Unbounded Trace - CELL 1

# Create trace connection, trace session, and start the trace

# Create the trace connection
trace_connection = fabric.create_trace_connection(workspace=target_workspace, dataset=target_semantic_model)

# Define the trace session
trace = trace_connection.create_trace(event_schema, "SemPyDAXQueryTrace")

# Start the trace
trace.start(5)

# Variable to count the duration of trace
trace_start_time = time.time()

print("Trace started. You can now interact with your report...")

In [ ]:
## Unbounded Trace - CELL 2

# Stop the trace, retrieve the logs, and clean up the session and connection

# Stop the trace and retrieve the logs
trace_logs = trace.stop()
duration = time.time() - trace_start_time

print(f"Trace complete after {round(duration, 0)} seconds")

# Clean up the trace and trace connection
trace.drop()
trace_connection.drop_trace("SemPyDAXQueryTrace")
trace_connection.disconnect_and_dispose()

display(trace_logs)